Random Forest

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/text_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # -------------------------
    # Drop missing values
    # -------------------------
    df_phase = df_phase.dropna()

    # -------------------------
    # Features and Labels
    # -------------------------
    X = df_phase[feature_cols]
    y = df_phase["label"]

    print("Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # -------------------------
    # Train-Test Split
    # -------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    # -------------------------
    # Feature Scaling
    # -------------------------
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # -------------------------
    # Random Forest Model
    # -------------------------
    model = RandomForestClassifier(
        n_estimators=400,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    # -------------------------
    # Predictions
    # -------------------------
    THRESHOLD = 0.40

    y_prob = model.predict_proba(X_test)[:, 1]

    y_pred = (y_prob > THRESHOLD).astype(int)

    # -------------------------
    # Results
    # -------------------------
    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # -------------------------
    # Classification Report
    # -------------------------
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # -------------------------
    # ROC-AUC
    # -------------------------
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # -------------------------
    # Confusion Matrix
    # -------------------------
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return model, roc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 780)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 777

TRAINING FOR PHASE 1
Shape: (141, 777)
Class Distribution: [99 42]

Predicted Distribution:
[28  1]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.71      1.00      0.83        20
           1       1.00      0.11      0.20         9

    accuracy                           0.72        29
   macro avg       0.86      0.56      0.52        29
weighted avg       0.80      0.72      0.64        29

ROC-AUC Score: 0.6083

Confusion Matrix:
[[20  0]
 [ 8  1]]

TRAINING FOR PHASE 2
Shape: (141, 777)
Class Distribution: [99 42]

Predicted Distribution:
[28  1]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.71      1.00      0.83        20
           1       1.00      0.11      0.

Random Forest 5 folds

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/text_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION (5-FOLD RF)
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # -------------------------
    # Drop missing values
    # -------------------------
    df_phase = df_phase.dropna()

    # -------------------------
    # Features and Labels
    # -------------------------
    X = df_phase[feature_cols].values
    y = df_phase["label"].values

    print("Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # 10-FOLD SPLIT
    # =====================================================
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # Store all predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        # -------------------------
        # Split Data
        # -------------------------
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # -------------------------
        # Feature Scaling
        # -------------------------
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # -------------------------
        # Random Forest Model
        # -------------------------
        model = RandomForestClassifier(
            n_estimators=400,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

        # -------------------------
        # Train
        # -------------------------
        model.fit(X_train, y_train)

        # -------------------------
        # Predict
        # -------------------------
        THRESHOLD = 0.40

        y_prob = model.predict_proba(X_test)[:, 1]

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # -------------------------
        # Fold ROC
        # -------------------------
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # FINAL CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 780)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 777

TRAINING FOR PHASE 1
Shape: (141, 777)
Class Distribution: [99 42]

----- Fold 1 -----
Fold ROC-AUC: 0.4556

----- Fold 2 -----
Fold ROC-AUC: 0.7812

----- Fold 3 -----
Fold ROC-AUC: 0.7531

----- Fold 4 -----
Fold ROC-AUC: 0.9375

----- Fold 5 -----
Fold ROC-AUC: 0.8041

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.4556 0.7812 0.7531 0.9375 0.8041]

Mean Fold ROC-AUC:
0.7463

Overall ROC-AUC:
0.7143

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.94      0.82        99
           1       0.57      0.19      0.29        42

    accuracy                           0.72       141
   macro avg       0.65      0.56      0.55       141
weighted avg       0.68      0.72      0.66       141


Confusion Matrix:
[[93  6]
 [34  8]]

TRAINING FOR PHASE 2
Shape: (141, 777)
Class D

Random Forest 10 folds

In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/text_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION (5-FOLD RF)
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # -------------------------
    # Drop missing values
    # -------------------------
    df_phase = df_phase.dropna()

    # -------------------------
    # Features and Labels
    # -------------------------
    X = df_phase[feature_cols].values
    y = df_phase["label"].values

    print("Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # 10-FOLD SPLIT
    # =====================================================
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # Store all predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        # -------------------------
        # Split Data
        # -------------------------
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # -------------------------
        # Feature Scaling
        # -------------------------
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # -------------------------
        # Random Forest Model
        # -------------------------
        model = RandomForestClassifier(
            n_estimators=400,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

        # -------------------------
        # Train
        # -------------------------
        model.fit(X_train, y_train)

        # -------------------------
        # Predict
        # -------------------------
        THRESHOLD = 0.40

        y_prob = model.predict_proba(X_test)[:, 1]

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # -------------------------
        # Fold ROC
        # -------------------------
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # FINAL CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 780)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 777

TRAINING FOR PHASE 1
Shape: (141, 777)
Class Distribution: [99 42]

----- Fold 1 -----
Fold ROC-AUC: 0.28

----- Fold 2 -----
Fold ROC-AUC: 0.6875

----- Fold 3 -----
Fold ROC-AUC: 0.7

----- Fold 4 -----
Fold ROC-AUC: 0.975

----- Fold 5 -----
Fold ROC-AUC: 0.65

----- Fold 6 -----
Fold ROC-AUC: 0.925

----- Fold 7 -----
Fold ROC-AUC: 0.925

----- Fold 8 -----
Fold ROC-AUC: 0.8375

----- Fold 9 -----
Fold ROC-AUC: 0.9

----- Fold 10 -----
Fold ROC-AUC: 0.8

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.28   0.6875 0.7    0.975  0.65   0.925  0.925  0.8375 0.9    0.8   ]

Mean Fold ROC-AUC:
0.768

Overall ROC-AUC:
0.7454

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.96      0.83        99
           1       0.67      0.19      0.30        42

    accuracy            

XGBoost

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/text_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"]

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    selector = VarianceThreshold(
        threshold=0.0001
    )

    X = selector.fit_transform(X)

    print("Shape After Variance Threshold:", X.shape)

    # =====================================================
    # TRAIN TEST SPLIT
    # =====================================================
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # =====================================================
    # HANDLE CLASS IMBALANCE
    # =====================================================
    neg = np.sum(y_train == 0)
    pos = np.sum(y_train == 1)

    scale_pos_weight = neg / pos

    print("Scale Pos Weight:", round(scale_pos_weight, 2))

    # =====================================================
    # OPTIMIZED XGBOOST MODEL
    # =====================================================
    model = XGBClassifier(

        # Core
        objective="binary:logistic",
        eval_metric="auc",

        # Smaller trees prevent overfit
        n_estimators=250,
        max_depth=3,

        # Learning
        learning_rate=0.05,

        # Sampling
        subsample=0.7,
        colsample_bytree=0.6,

        # Regularization
        min_child_weight=5,
        gamma=2,

        # Imbalance handling
        scale_pos_weight=scale_pos_weight,

        # Misc
        random_state=42,
        n_jobs=-1
    )

    # =====================================================
    # TRAIN MODEL
    # =====================================================
    model.fit(X_train, y_train)

    # =====================================================
    # PREDICT PROBABILITIES
    # =====================================================
    y_prob = model.predict_proba(X_test)[:, 1]

    # =====================================================
    # THRESHOLD
    # =====================================================
    THRESHOLD = 0.35

    y_pred = (y_prob > THRESHOLD).astype(int)

    # =====================================================
    # RESULTS
    # =====================================================
    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # =====================================================
    # ROC-AUC
    # =====================================================
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return model, roc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 780)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 777

TRAINING FOR PHASE 1
Original Shape: (141, 777)
Class Distribution: [99 42]
Shape After Variance Threshold: (141, 352)
Scale Pos Weight: 2.39

Predicted Distribution:
[22  7]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.80      0.76        20
           1       0.43      0.33      0.38         9

    accuracy                           0.66        29
   macro avg       0.58      0.57      0.57        29
weighted avg       0.63      0.66      0.64        29

ROC-AUC Score: 0.7389

Confusion Matrix:
[[16  4]
 [ 6  3]]

TRAINING FOR PHASE 2
Original Shape: (141, 777)
Class Distribution: [99 42]
Shape After Variance Threshold: (141, 312)
Scale Pos Weight: 2.39

Predicted Distribution:
[19 10]

Actual Distribution:
[20  9]

Classification Report:


XGBoost 5 folds

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import VarianceThreshold

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/text_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    selector = VarianceThreshold(
        threshold=0.0001
    )

    X = selector.fit_transform(X)

    print("Shape After Variance Threshold:", X.shape)

    # =====================================================
    # CLASS IMBALANCE
    # =====================================================
    neg = np.sum(y == 0)
    pos = np.sum(y == 1)

    scale_pos_weight = neg / pos

    print("Scale Pos Weight:", round(scale_pos_weight, 2))

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # XGBOOST MODEL
        # =================================================
        model = XGBClassifier(

            objective="binary:logistic",
            eval_metric="auc",

            # Trees
            n_estimators=250,
            max_depth=3,

            # Learning
            learning_rate=0.05,

            # Sampling
            subsample=0.7,
            colsample_bytree=0.6,

            # Regularization
            min_child_weight=5,
            gamma=2,

            # Imbalance
            scale_pos_weight=scale_pos_weight,

            # Misc
            random_state=42,
            n_jobs=-1
        )

        # =================================================
        # TRAIN
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.35

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store fold predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 780)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 777

TRAINING FOR PHASE 1
Original Shape: (141, 777)
Class Distribution: [99 42]
Shape After Variance Threshold: (141, 352)
Scale Pos Weight: 2.36

----- Fold 1 -----
Fold ROC-AUC: 0.3944

----- Fold 2 -----
Fold ROC-AUC: 0.7625

----- Fold 3 -----
Fold ROC-AUC: 0.6938

----- Fold 4 -----
Fold ROC-AUC: 0.7375

----- Fold 5 -----
Fold ROC-AUC: 0.6959

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.3944 0.7625 0.6938 0.7375 0.6959]

Mean Fold ROC-AUC:
0.6568

Overall ROC-AUC:
0.6496

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.67      0.71        99
           1       0.38      0.48      0.42        42

    accuracy                           0.61       141
   macro avg       0.56      0.57      0.56       141
weighted avg       0.64      0.61      0.62       141


Confusion

XGBoost 10 folds

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import VarianceThreshold

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/text_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    selector = VarianceThreshold(
        threshold=0.0001
    )

    X = selector.fit_transform(X)

    print("Shape After Variance Threshold:", X.shape)

    # =====================================================
    # CLASS IMBALANCE
    # =====================================================
    neg = np.sum(y == 0)
    pos = np.sum(y == 1)

    scale_pos_weight = neg / pos

    print("Scale Pos Weight:", round(scale_pos_weight, 2))

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # XGBOOST MODEL
        # =================================================
        model = XGBClassifier(

            objective="binary:logistic",
            eval_metric="auc",

            # Trees
            n_estimators=250,
            max_depth=3,

            # Learning
            learning_rate=0.05,

            # Sampling
            subsample=0.7,
            colsample_bytree=0.6,

            # Regularization
            min_child_weight=5,
            gamma=2,

            # Imbalance
            scale_pos_weight=scale_pos_weight,

            # Misc
            random_state=42,
            n_jobs=-1
        )

        # =================================================
        # TRAIN
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.35

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store fold predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 780)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 777

TRAINING FOR PHASE 1
Original Shape: (141, 777)
Class Distribution: [99 42]
Shape After Variance Threshold: (141, 352)
Scale Pos Weight: 2.36

----- Fold 1 -----
Fold ROC-AUC: 0.4

----- Fold 2 -----
Fold ROC-AUC: 0.5

----- Fold 3 -----
Fold ROC-AUC: 0.825

----- Fold 4 -----
Fold ROC-AUC: 0.925

----- Fold 5 -----
Fold ROC-AUC: 0.75

----- Fold 6 -----
Fold ROC-AUC: 0.725

----- Fold 7 -----
Fold ROC-AUC: 0.85

----- Fold 8 -----
Fold ROC-AUC: 0.7

----- Fold 9 -----
Fold ROC-AUC: 0.575

----- Fold 10 -----
Fold ROC-AUC: 0.7778

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.4    0.5    0.825  0.925  0.75   0.725  0.85   0.7    0.575  0.7778]

Mean Fold ROC-AUC:
0.7028

Overall ROC-AUC:
0.6859

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.71      0.75        99
    

SVM

In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/text_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"]

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    selector = VarianceThreshold(
        threshold=0.001
    )

    X = selector.fit_transform(X)

    print("Shape After Variance Threshold:", X.shape)

    # =====================================================
    # TRAIN TEST SPLIT
    # =====================================================
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # =====================================================
    # FEATURE SCALING
    # =====================================================
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # =====================================================
    # LINEAR SVM MODEL
    # =====================================================
    model = SVC(

        # Linear works better for
        # high-dimensional embeddings
        kernel="linear",

        # Needed for ROC-AUC
        probability=True,

        # Handle imbalance
        class_weight="balanced",

        # Better generalization
        C=0.5,

        random_state=42
    )

    # =====================================================
    # TRAIN MODEL
    # =====================================================
    model.fit(X_train, y_train)

    # =====================================================
    # PREDICT PROBABILITIES
    # =====================================================
    y_prob = model.predict_proba(X_test)[:, 1]

    # =====================================================
    # THRESHOLD
    # =====================================================
    THRESHOLD = 0.35

    y_pred = (y_prob > THRESHOLD).astype(int)

    # =====================================================
    # RESULTS
    # =====================================================
    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # =====================================================
    # ROC-AUC
    # =====================================================
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return model, roc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 780)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 777

TRAINING FOR PHASE 1
Original Shape: (141, 777)
Class Distribution: [99 42]
Shape After Variance Threshold: (141, 8)

Predicted Distribution:
[26  3]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.65      0.85      0.74        20
           1       0.00      0.00      0.00         9

    accuracy                           0.59        29
   macro avg       0.33      0.42      0.37        29
weighted avg       0.45      0.59      0.51        29

ROC-AUC Score: 0.5111

Confusion Matrix:
[[17  3]
 [ 9  0]]

TRAINING FOR PHASE 2
Original Shape: (141, 777)
Class Distribution: [99 42]
Shape After Variance Threshold: (141, 8)

Predicted Distribution:
[28  1]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   supp

SVM 5 folds

In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/text_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    selector = VarianceThreshold(
        threshold=0.001
    )

    X = selector.fit_transform(X)

    print("Shape After Variance Threshold:", X.shape)

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # FEATURE SCALING
        # =================================================
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # =================================================
        # SVM MODEL
        # =================================================
        model = SVC(

            kernel="linear",

            probability=True,

            class_weight="balanced",

            C=0.5,

            random_state=42
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.35

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 780)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 777

TRAINING FOR PHASE 1
Original Shape: (141, 777)
Class Distribution: [99 42]
Shape After Variance Threshold: (141, 8)

----- Fold 1 -----
Fold ROC-AUC: 0.7389

----- Fold 2 -----
Fold ROC-AUC: 0.625

----- Fold 3 -----
Fold ROC-AUC: 0.2688

----- Fold 4 -----
Fold ROC-AUC: 0.5875

----- Fold 5 -----
Fold ROC-AUC: 0.4971

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.7389 0.625  0.2688 0.5875 0.4971]

Mean Fold ROC-AUC:
0.5434

Overall ROC-AUC:
0.5842

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.93      0.82        99
           1       0.53      0.19      0.28        42

    accuracy                           0.71       141
   macro avg       0.63      0.56      0.55       141
weighted avg       0.67      0.71      0.66       141


Confusion Matrix:
[[92  7]
 [34  8]

SVM 10 folds

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/text_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    selector = VarianceThreshold(
        threshold=0.001
    )

    X = selector.fit_transform(X)

    print("Shape After Variance Threshold:", X.shape)

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # FEATURE SCALING
        # =================================================
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # =================================================
        # SVM MODEL
        # =================================================
        model = SVC(

            kernel="linear",

            probability=True,

            class_weight="balanced",

            C=0.5,

            random_state=42
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.35

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 780)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 777

TRAINING FOR PHASE 1
Original Shape: (141, 777)
Class Distribution: [99 42]
Shape After Variance Threshold: (141, 8)

----- Fold 1 -----
Fold ROC-AUC: 0.74

----- Fold 2 -----
Fold ROC-AUC: 0.725

----- Fold 3 -----
Fold ROC-AUC: 0.5

----- Fold 4 -----
Fold ROC-AUC: 0.25

----- Fold 5 -----
Fold ROC-AUC: 0.725

----- Fold 6 -----
Fold ROC-AUC: 0.75

----- Fold 7 -----
Fold ROC-AUC: 0.55

----- Fold 8 -----
Fold ROC-AUC: 0.625

----- Fold 9 -----
Fold ROC-AUC: 0.3

----- Fold 10 -----
Fold ROC-AUC: 0.4667

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.74   0.725  0.5    0.25   0.725  0.75   0.55   0.625  0.3    0.4667]

Mean Fold ROC-AUC:
0.5632

Overall ROC-AUC:
0.5435

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.76      0.73        99
           1       0.31      